- 基于时序差分（Temporal-Difference, TD）的方法：利用值函数（Value Function）来估计优势，考虑每个时间步（token）的奖励，例如 `GAE`。
- 基于蒙特卡洛（Monte Carlo）的方法：使用完整的序列奖励（outcome reward），通常会结合各种基线（baseline）来减小方差，例如 GRPO, RLOO 等大部分算法。这类算法的优势函数在单个序列的所有 token 上通常是常数。
- Token 级别优势估计：这类方法为序列中的每一步（Token）计算优势 $A_t$，通常依赖于价值函数 $V_t$ 或者蒙特卡洛回报 $G_t$
    - GAE
    - REINFORCE++ (RF++)
    - REMAX
- 序列级别（结果导向）优势估计 (Outcome-Based)
    - GRPO
    - REINFORCE++-Baseline
    - RLOO (Reinforcement Learning with Leave-One-Out)
 

### 批次标准化（batch whitening）与组内标准化

> 在强化学习（RL），特别是策略梯度算法（如 PPO 及其变体）中，对优势（Advantage）进行归一化处理是稳定训练、降低梯度方差的关键步骤。其目的通常是调整优势值的尺度，使其分布更易于学习。
- “批次标准化”和“组内标准化”是两种核心策略。它们的核心区别在于计算归一化所需的统计量（均值和标准差）时所使用的数据范围（Scope）。
    - 批次（Batch）：指一次训练迭代中使用的全部数据样本。
    - 组（Group）：指批次中具有共同输入（例如，同一个 Prompt $q$）的所有响应样本集合。

| 样本 ID | 组 (Prompt) | 奖励 (R) |
|---|---|---|
| 1 | A (简单任务) | 10 |
| 2 | A (简单任务) | 8 |
| 3 | B (困难任务) | 2 |
| 4 | B (困难任务) | 0 |

- 组内归一化
    - $\mu_A=9,\sigma_A=1;\mu_B=1,\sigma_B=1$
    - 标准化后优势:
        - 样本1：(10-9)/1=1
        - 样本2：(8-9)/1=-1
        - 样本3：(2-1)/1=1
        - 样本4：(0-1)/1=-1
- 批次归一化
    - $\mu_{Batch}=5,\sigma_{Batch}=4.12$
- 批次标准化：强调全局表现。优势值表示“该响应相对于批次中所有响应（无论来自哪个 Prompt）的平均水平有多好”。
    - 组内标准化：强调局部排序（Intra-prompt Ranking）。优势值表示“该响应相对于同一个 Prompt 的其他响应有多好”。这对于需要从多个候选中选择最佳响应的任务（如 RLHF 中的偏好学习）尤其重要。
- 基线（Baseline）的定义: 在策略梯度中，中心化相当于引入了一个基线来降低方差。
    - 组内中心化：使用的是一个依赖于输入状态（Prompt）的局部基线（$\mu_x$）。这种状态相关的基线通常比全局基线能更有效地降低方差。
    - 批次中心化：使用的是一个全局基线（$\mu_{Batch}$）。

### GAE

### REINFORCE++, REINFORCE++-bl

- RF++
    - 不依赖分组，它直接使用蒙特卡洛回报（discounted future returns）作为优势，并对其进行 whiten 处理。
    - 时间步 $t$，回报 $R_t=\sum_{k=t}^{T-1}\gamma^{k-t}r_k$
    - $\hat A_t=\text{whiten}(R_t)$
- RF++-bl
    - 该算法与 GRPO 非常相似，也使用组内平均奖励作为基线。主要区别在于计算出 (序列奖励 - 组均值) 后，会进行 whiten 操作

In [1]:
import numpy as np

In [3]:
token_level_rewards = np.zeros((5, 8))

In [6]:
token_level_rewards[:, -1] = np.random.randint(0, 2, 5)

In [7]:
token_level_rewards

array([[0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 0., 1.]])

In [14]:
gamma = 0.9
running_return = 0
returns = np.zeros_like(token_level_rewards)
response_mask = np.ones_like(token_level_rewards)
for t in reversed(range(token_level_rewards.shape[1])):
    running_return = token_level_rewards[:, t] + gamma * running_return
    returns[:, t] = running_return
    # Reset after EOS
    running_return = running_return * response_mask[:, t]
returns

array([[0.4782969, 0.531441 , 0.59049  , 0.6561   , 0.729    , 0.81     ,
        0.9      , 1.       ],
       [0.4782969, 0.531441 , 0.59049  , 0.6561   , 0.729    , 0.81     ,
        0.9      , 1.       ],
       [0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
        0.       , 0.       ],
       [0.4782969, 0.531441 , 0.59049  , 0.6561   , 0.729    , 0.81     ,
        0.9      , 1.       ],
       [0.4782969, 0.531441 , 0.59049  , 0.6561   , 0.729    , 0.81     ,
        0.9      , 1.       ]])

### GRPO vs. RLOO